# 04 — Tree Model Improvement

## Objective

In the previous modeling notebook, simple linear models performed poorly, while tree-based models produced the first meaningful results.

The Random Forest model achieved the best performance so far.

The goal of this notebook is to improve tree-based model performance in a structured and explainable way.

---

# Why This Notebook Exists

Notebook 03 showed that:

- Linear Regression was unstable
- Ridge Regression improved stability but still performed poorly
- Decision Trees performed better than linear models
- Random Forest produced the strongest result so far

This suggests that the dataset likely contains:

- non-linear relationships
- feature interactions
- sparse activation patterns

Because tree-based models handled these patterns better, this notebook focuses on improving them.

---

# Improvement Strategy

We will improve tree-based models using three main ideas:

## 1. Rebuild the Random Forest Baseline

We first recreate the Random Forest model from the previous notebook.

This gives us a reference point for comparison.

---

## 2. Add Feature Engineering

Because the dataset has anonymous sparse features, we create row-level statistical features such as:

- number of non-zero values
- row sum
- row mean
- row standard deviation
- row maximum value

These features may summarize useful structural information.

---

## 3. Tune Model Hyperparameters

We then adjust Random Forest settings such as:

- number of trees
- tree depth
- minimum samples per leaf
- number of features considered at each split

The goal is to improve performance while avoiding overfitting.

---

# Expected Outcome

By the end of this notebook, we should understand:

- whether engineered row-level features improve performance
- whether Random Forest tuning improves performance
- which settings produce the most stable validation results
- whether tree-based models remain the best direction for this dataset

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import KFold, cross_validate, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.max_columns", 100)

RANDOM_STATE = 42

# 1. Load Data and Recreate Baseline Setup

We begin by loading the same dataset and applying the same basic preprocessing used in the previous notebook.

This ensures that results in this notebook are comparable to the previous Random Forest result.

The steps are:

- load train and test data
- separate features and target
- remove ID columns
- remove constant features
- recreate the same KFold validation setup

In [2]:
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

y = train["target"]
X = train.drop(columns=["ID", "target"])
X_test = test.drop(columns=["ID"])

feature_variances = X.var()
constant_cols = feature_variances[feature_variances == 0].index

X = X.drop(columns=constant_cols)
X_test = X_test.drop(columns=constant_cols)

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

print("Removed constant features:", len(constant_cols))
print("X shape:", X.shape)
print("X_test shape:", X_test.shape)

Removed constant features: 256
X shape: (4459, 4735)
X_test shape: (49342, 4735)


# 2. Rebuild Random Forest Baseline

Before improving the model, we first recreate the Random Forest baseline from the previous notebook.

This gives us a reference point.

Any feature engineering or tuning should be compared against this baseline.

In [3]:
baseline_forest = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

baseline_scores = cross_validate(
    baseline_forest,
    X,
    y,
    cv=kf,
    scoring=(
        "neg_mean_absolute_error",
        "neg_root_mean_squared_error",
        "r2"
    ),
    return_train_score=True
)

baseline_mae = -baseline_scores["test_neg_mean_absolute_error"]
baseline_rmse = -baseline_scores["test_neg_root_mean_squared_error"]
baseline_r2 = baseline_scores["test_r2"]

print("Baseline Random Forest")
print("MAE:", baseline_mae.mean())
print("RMSE:", baseline_rmse.mean())
print("R²:", baseline_r2.mean())

Baseline Random Forest
MAE: 5130281.796040111
RMSE: 7302710.862647332
R²: 0.20820847651407579


# 3. Feature Engineering for Tree Models

The previous EDA notebooks showed that the dataset is highly sparse and that rows differ significantly in the number of active features.

Instead of relying only on individual anonymous columns, we now create row-level statistical features that summarize the structure of each observation.

The goal is to determine whether these aggregate features help tree-based models detect additional patterns.

We will create features such as:

- number of non-zero values
- row sum
- row mean
- row standard deviation
- row maximum value
- row minimum value

These features may capture information about overall row behavior that individual features alone do not provide.

In [4]:
X_engineered = X.copy()
X_test_engineered = X_test.copy()

# Number of active (non-zero) features
X_engineered["row_nonzero_count"] = (X != 0).sum(axis=1)
X_test_engineered["row_nonzero_count"] = (X_test != 0).sum(axis=1)

# Row statistics
X_engineered["row_sum"] = X.sum(axis=1)
X_test_engineered["row_sum"] = X_test.sum(axis=1)

X_engineered["row_mean"] = X.mean(axis=1)
X_test_engineered["row_mean"] = X_test.mean(axis=1)

X_engineered["row_std"] = X.std(axis=1)
X_test_engineered["row_std"] = X_test.std(axis=1)

X_engineered["row_max"] = X.max(axis=1)
X_test_engineered["row_max"] = X_test.max(axis=1)

X_engineered["row_min"] = X.min(axis=1)
X_test_engineered["row_min"] = X_test.min(axis=1)

print("Original shape:", X.shape)
print("Engineered shape:", X_engineered.shape)

Original shape: (4459, 4735)
Engineered shape: (4459, 4741)


# 4. Random Forest With Engineered Features

We now retrain the Random Forest model using the engineered row-level features.

The goal is to determine whether structural row summaries improve predictive performance compared to the original baseline model.

In [5]:
engineered_forest = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

engineered_scores = cross_validate(
    engineered_forest,
    X_engineered,
    y,
    cv=kf,
    scoring=(
        "neg_mean_absolute_error",
        "neg_root_mean_squared_error",
        "r2"
    ),
    return_train_score=True
)

engineered_mae = -engineered_scores["test_neg_mean_absolute_error"]
engineered_rmse = -engineered_scores["test_neg_root_mean_squared_error"]
engineered_r2 = engineered_scores["test_r2"]

print("Random Forest With Engineered Features")
print("MAE:", engineered_mae.mean())
print("RMSE:", engineered_rmse.mean())
print("R²:", engineered_r2.mean())

Random Forest With Engineered Features
MAE: 4665446.938568786
RMSE: 6902573.465768479
R²: 0.2920077873292265


# Feature Engineering Results

Adding row-level engineered features improved Random Forest performance across all evaluation metrics.

Compared to the baseline model:

- MAE decreased
- RMSE decreased
- R² increased from approximately 0.21 to 0.29

This suggests that aggregate row statistics contain useful predictive information beyond the original anonymous feature columns.

The improvement also supports findings from the EDA notebooks:

- sparsity structure matters
- row activity patterns are informative
- global row behavior may contain predictive signal

These results show that even simple engineered features can substantially improve tree-based model performance.

# 5. Hyperparameter Tuning

We now investigate whether adjusting Random Forest hyperparameters can further improve predictive performance.

The baseline model used relatively simple default settings.

However, tree-based models are highly sensitive to parameters such as:

- tree depth
- number of trees
- minimum samples per split
- minimum samples per leaf
- feature subsampling

Careful tuning may improve generalization and reduce overfitting.

In [6]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [10, 20],
    "min_samples_leaf": [1, 5],
}

tuned_forest = RandomForestRegressor(
    random_state=RANDOM_STATE,
    n_jobs=-1
)

grid_search = GridSearchCV(
    estimator=tuned_forest,
    param_grid=param_grid,
    cv=kf,
    scoring="r2",
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_engineered, y)

print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest CV R²:")
print(grid_search.best_score_)

Fitting 5 folds for each of 8 candidates, totalling 40 fits
Best Parameters:
{'max_depth': 20, 'min_samples_leaf': 5, 'n_estimators': 200}

Best CV R²:
0.3220338201293507
